#### 1. Create your own GitHub repository and a GitHub webpage where you can include information about you and your data projects and research interests. A GitHub webpage should be visible via an internet browser at the address yourname.github.io/projectname where "your name" represents your user name for GitHub and "project name" is the name you choose for your project/webspace.

#### 2. Coding problem: implement a gradient descent method for Ridge Regression by using the PyTorch library. Your implementation should be a class that has the required methods “.fit” and “.predict”. You should include an application of your code to a data set.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Libraries of functions need to be imported
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn import linear_model
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.spatial import Delaunay
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_squared_error as mse
from scipy import linalg
from sklearn.model_selection import train_test_split as tts
from scipy.interpolate import interp1d, LinearNDInterpolator, NearestNDInterpolator
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error, r2_score

# the following line(s) are necessary if you want to make SKlearn compliant functions
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted

#Imported lines from Efficient_ and Applications with


In [ ]:
data = pd.read_csv('drive/MyDrive/Gelila_WM/DATA_441/Data_Sets/cars.csv')

In [ ]:
data.head()

,MPG,CYL,ENG,WGT
0,18.0,8,307.0,3504
1,15.0,8,350.0,3693
2,18.0,8,318.0,3436
3,16.0,8,304.0,3433
4,17.0,8,302.0,3449


In [ ]:
X = data[['CYL', 'ENG', 'WGT']]
y = data['MPG']

In [ ]:
X

,CYL,ENG,WGT
0,8,307.0,3504
1,8,350.0,3693
2,8,318.0,3436
3,8,304.0,3433
4,8,302.0,3449
...,...,...,...
387,4,140.0,2790
388,4,97.0,2130
389,4,135.0,2295
390,4,120.0,2625


In [ ]:
y

0      18.0
1      15.0
2      18.0
3      16.0
4      17.0
       ... 
387    27.0
388    44.0
389    32.0
390    28.0
391    31.0
Name: MPG, Length: 392, dtype: float64

In [ ]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

class RidgeReg:
    def __init__(self, n_features, learning_rate=0.005, regularization_param=0.01, epochs=1500):
        self.n_features = n_features
        self.learning_rate = learning_rate
        self.regularization_param = regularization_param
        self.epochs = epochs
        self.weights = np.random.randn(n_features + 1, 1)
        self.losses = []

    def fit(self, X, y):
        X = np.hstack((np.ones((X.shape[0], 1)), X))
        for _ in range(self.epochs):
            outputs = np.dot(X, self.weights)
            loss = np.mean((outputs - y) ** 2)
            regularization_loss = np.sum(self.weights[1:] ** 2)
            total_loss = loss + self.regularization_param * regularization_loss
            gradient = 2 * np.dot(X.T, (outputs - y)) / X.shape[0]
            gradient[1:] += 2 * self.regularization_param * self.weights[1:]
            self.weights -= self.learning_rate * gradient
            self.losses.append(total_loss)

    def predict(self, X):
        X = np.hstack((np.ones((X.shape[0], 1)), X))
        return np.dot(X, self.weights)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

poly = PolynomialFeatures(degree=2, include_bias=False)
X_scaled_poly = poly.fit_transform(X_scaled)

X_train, X_test, y_train, y_test = train_test_split(X_scaled_poly, y, test_size=0.2, random_state=42)

y_train = np.array(y_train).reshape(-1, 1)
y_test = np.array(y_test).reshape(-1, 1)

model = RidgeReg(n_features=X_train.shape[1], learning_rate=0.005, regularization_param=0.01, epochs=1500)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, predictions)
r_squared = r2_score(y_test, predictions)

print("MSE:", mse)
print("RMSE:", rmse)
print("MAE:", mae)
print("R-squared:", r_squared)


MSE: 17.889604165670754
RMSE: 4.229610403532547
MAE: 3.191662794365368
R-squared: 0.6495023459706439


#### 3. Complete the exercise provided in the Application to Locally Weighted Regression notebook and test the method on a data set, for example, the one provided in class.

Exercise:
Adjust the code below and make it work without errors. Compare the results with the previous ones.

In [ ]:
class Lowess:
    def __init__(self, kernel = Gaussian, tau=0.05):
        self.kernel = kernel
        self.tau = tau

    def fit(self, x, y):
        kernel = self.kernel
        tau = self.tau
        # w = weights_matrix(x,x,kernel,tau)
        # if np.isscalar(x):
        #   lm.fit(np.diag(w).dot(x.reshape(-1,1)),np.diag(w).dot(y.reshape(-1,1)))
        #   yest = lm.predict([[x]])[0][0]
        # else:
        #   n = len(x)
        #   yest = np.zeros(n)
        #   #Looping through all x-points
        #   for i in range(n):
        #     lm.fit(np.diag(w[i,:]).dot(x.reshape(-1,1)),np.diag(w[i,:]).dot(y.reshape(-1,1)))
        #     yest[i] = lm.predict(x[i].reshape(-1,1))
        self.xtrain_ = x
        self.yhat_ = y

    def predict(self, x_new):
        check_is_fitted(self)
        x = self.xtrain_
        y = self.yhat_
        lm = linear_model.Ridge(alpha=0.001)
        w = weights_matrix(x,x_new,self.kernel,self.tau)

        if np.isscalar(x_new):
          lm.fit(np.diag(w)@(x.reshape(-1,1)),np.diag(w)@(y.reshape(-1,1)))
          yest = lm.predict([[x_new]])[0][0]
        else:
          n = len(x_new)
          yest_test = np.zeros(n)
          #Looping through all x-points
          for i in range(n):
            lm.fit(np.diag(w[i,:])@x,np.diag(w[i,:])@y)
            yest_test[i] = lm.predict(x_new[i].reshape(1,-1))
        return yest_test

In [ ]:
model = Lowess(kernel=Gaussian,tau=0.05)
model.fit(xscaled,y)
model.predict(xscaled)


In [ ]:
mse(y,model.predict(xscaled))

##My Answer

In [ ]:
X

,CYL,ENG,WGT
0,8,307.0,3504
1,8,350.0,3693
2,8,318.0,3436
3,8,304.0,3433
4,8,302.0,3449
...,...,...,...
387,4,140.0,2790
388,4,97.0,2130
389,4,135.0,2295
390,4,120.0,2625


In [ ]:
X =X.values
X

array([[   8.,  307., 3504.],
       [   8.,  350., 3693.],
       [   8.,  318., 3436.],
       ...,
       [   4.,  135., 2295.],
       [   4.,  120., 2625.],
       [   4.,  119., 2720.]])

In [ ]:
y=y.values

In [ ]:
y

array([18.      , 15.      , 18.      , 16.      , 17.      , 15.      ,
       14.      , 14.      , 14.      , 15.      , 15.      , 14.      ,
       15.      , 14.      , 24.      , 22.      , 18.      , 21.      ,
       27.      , 26.      , 25.      , 24.      , 25.      , 26.      ,
       21.      , 10.      , 10.      , 11.      ,  9.      , 27.      ,
       28.      , 25.      , 19.      , 16.      , 17.      , 19.      ,
       18.      , 14.      , 14.      , 14.      , 14.      , 12.      ,
       13.      , 13.      , 18.      , 22.      , 19.      , 18.      ,
       23.      , 28.      , 30.      , 30.      , 31.      , 35.      ,
       27.      , 26.      , 24.      , 25.      , 23.      , 20.      ,
       21.      , 13.      , 14.      , 15.      , 14.      , 17.      ,
       11.      , 13.      , 12.      , 13.      , 19.      , 15.      ,
       13.      , 13.      , 14.      , 18.      , 22.      , 21.      ,
       26.      , 22.      , 28.      , 23.      , 

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
#All the functions

# Gaussian Kernel
def Gaussian(x):
  return np.where(np.abs(x)>4,0,1/(np.sqrt(2*np.pi))*np.exp(-1/2*x**2))

# Quartic Kernel
def Quartic(x):
  return np.where(np.abs(x)>1,0,15/16*(1-np.abs(x)**2)**2)

# Epanechnikov Kernel
def Epanechnikov(x):
  return np.where(np.abs(x)>1,0,3/4*(1-np.abs(x)**2))
  # this is the correct vectorized version
def Tricubic(x):
  return np.where(np.abs(x)>1,0,(1-np.abs(x)**3)**3)

def dist(u,v):
  if len(v.shape)==1:
    v = v.reshape(1,-1)
    d = np.sqrt(np.sum((u-v)**2,axis=1))
  else:
    d = np.array([np.sqrt(np.sum((u-v[i])**2,axis=1)) for i in range(len(v))])
  return d

def kernel_function(xi,x0,kern, tau):
    return kern(dist(xi,x0)/(2*tau))

def weights_matrix(x,x_new,kern,tau):
  if np.isscalar(x_new):
    w = kernel_function(x,x_new,kern,tau)
  else:
    if len(x_new.shape)==1:
      n = 1
      w = kernel_function(x,x_new,kern,tau=0.05).reshape(1,-1)
    else:
      n = len(x_new)
      w = np.array([kernel_function(x,x_new[i],kern,tau) for i in range(n)]).reshape(n,len(x))
  return w

class Lowess:
    def __init__(self, kernel=Gaussian, tau=0.05):
        self.kernel = kernel
        self.tau = tau

    def fit(self, x, y):
        self.xtrain_ = x
        self.yhat_ = y

    def predict(self, x_new):
        check_is_fitted(self)
        x = self.xtrain_
        y = self.yhat_
        tau = self.tau

        if np.isscalar(x_new):
            distances = dist(x, np.array([x_new]))
            w = self.kernel(distances / (2 * tau))
            w_sum = np.sum(w)
            if w_sum == 0:  # Prevent division by zero
                w = np.ones_like(w)  # Set all weights to 1
            else:
                w /= w_sum
            lm = linear_model.Ridge(alpha=0.001)
            lm.fit(x.reshape(-1, 1), y, sample_weight=w.flatten())  # Flatten the weights to 1D array
            yest = lm.predict([[x_new]])[0]
            return yest
        else:
            n = len(x_new)
            yest_test = np.zeros(n)
            for i in range(n):
                distances = dist(x, np.array([x_new[i]]))
                w = self.kernel(distances / (2 * tau))
                w_sum = np.sum(w)
                if w_sum == 0:  # Prevent division by zero
                    w = np.ones_like(w)  # Set all weights to 1
                else:
                    w /= w_sum
                lm = linear_model.Ridge(alpha=0.001)
                lm.fit(x, y, sample_weight=w.flatten())  # Flatten the weights to 1D array
                yest_test[i] = lm.predict([x_new[i]])
            return yest_test


In [ ]:
type(X)

numpy.ndarray

In [ ]:
mse_lwr = []
r_squared = []
ep_mse = []
r_squaredE = []

kf = KFold(n_splits=10,shuffle=True,random_state=1234)
lowess_model = Lowess(kernel=Gaussian, tau=0.05)
model_lw = Lowess(kernel=Epanechnikov, tau=0.14)



for train_idx, test_idx in kf.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Fit and predict for Gaussian kernel
    lowess_model.fit(X_train, y_train)
    y_pred_gaussian = lowess_model.predict(X_test)

    mse_lwr.append(mean_squared_error(y_test, y_pred_gaussian))
    r_squared.append(r2_score(y_test, y_pred_gaussian))

    # Fit and predict for Epanechnikov kernel
    model_lw.fit(X_train, y_train)
    y_pred_epanechnikov = model_lw.predict(X_test)

    ep_mse.append(mean_squared_error(y_test, y_pred_epanechnikov))
    r_squaredE.append(r2_score(y_test, y_pred_epanechnikov))

print('The Cross-validated Mean Squared Error for Locally Weighted Regression (Gaussian Kernel) is:', np.mean(mse_lwr))
print('The mean R-squared (Gaussian Kernel):', np.mean(r_squared))

print('The Cross-validated Mean Squared Error for Locally Weighted Regression (Epanechnikov Kernel) is:', np.mean(ep_mse))
print('The mean R-squared (Epanechnikov Kernel):', np.mean(r_squaredE))

The Cross-validated Mean Squared Error for Locally Weighted Regression (Gaussian Kernel) is: 18.918041206041057
The mean R-squared (Gaussian Kernel): 0.6816278249772537
The Cross-validated Mean Squared Error for Locally Weighted Regression (Epanechnikov Kernel) is: 18.918041206041057
The mean R-squared (Epanechnikov Kernel): 0.6816278249772537
